# Planning an email test: a 12-version orthogonal array

**Design of Orthogonal Arrays (Taguchi Designs)** · a worked example from the StatNex teaching guide

**The story.** Before its next email campaign, a marketing team plans a test of four email features: a subject line (question or statement), three send times (morning, noon, evening), an image (none or product photo) and a button colour (red or blue).

**The question.** What is the smallest balanced set of email versions that tests all four features?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `05_anova/orthogonal_array_design.csv` (3 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Subject line,Send time,Image,Button colour
Question,Morning,No image,Red
Statement,Noon,Product photo,Blue
,Evening,,

These cells save `orthogonal_array.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile orthogonal_array.py
"""Orthogonal arrays, built rather than looked up in a table.

NumPy and SciPy have no orthogonal-array function, so the construction is written out
here. An array is not unique -- many different tables have the same size and the same
balance -- and this is the one StatNex builds:

* factors with the same number of levels share one array: the Rao-Hamming array over
  the finite field GF(s) for s = 2, 3, 4, 5, or Plackett-Burman's 12 runs when that is
  smaller for two-level factors;
* the arrays for different level counts are crossed, smallest level count first.
"""

import itertools

import numpy as np

# GF(4) is the one field in range that is not the integers mod s.
GF4_ADD = np.array([[0, 1, 2, 3], [1, 0, 3, 2], [2, 3, 0, 1], [3, 2, 1, 0]])
GF4_MUL = np.array([[0, 0, 0, 0], [0, 1, 2, 3], [0, 2, 3, 1], [0, 3, 1, 2]])


def field(level):
    """Addition and multiplication tables of GF(level), or None when there is no field."""
    if level == 4:
        return GF4_ADD, GF4_MUL
    if level in (2, 3, 5):
        index = np.arange(level)
        return (index[:, None] + index) % level, (index[:, None] * index) % level
    return None


def galois_array(level, power):
    """Rao-Hamming: level**power runs and (level**power - 1) / (level - 1) columns.

    The rows are every vector of length `power` over GF(level); the columns are the
    directions (vectors whose first non-zero entry is 1). Each entry is the inner product
    of the two, so any two columns take every pair of values equally often.

    A design with fewer factors takes the first columns, so the order matters: the basic
    directions first, then those mixing the most of them. A factor on a column that mixes
    w basic columns is tangled with their w-factor interaction, and the higher w, the
    less that matters. Four two-level factors in 8 runs thus get D = ABC, not C = AB.
    """
    add, mul = field(level)
    directions = [v for v in itertools.product(range(level), repeat=power)
                  if next((x for x in v if x), None) == 1]
    directions.sort(key=lambda d: (sum(map(bool, d)) != 1, -sum(map(bool, d))))
    rows = []
    for point in itertools.product(range(level), repeat=power):
        row = []
        for direction in directions:
            total = 0
            for a, b in zip(point, direction):
                total = add[total, mul[a, b]]
            row.append(total)
        rows.append(row)
    return np.array(rows) + 1


def plackett_burman_12():
    """The 12-run two-level design: a generator row rotated 11 times, plus a row of 2s."""
    generator = np.array([1, 1, -1, 1, 1, 1, -1, -1, -1, 1, -1])
    matrix = np.vstack([*[np.roll(generator, i) for i in range(11)], -np.ones(11)])
    return (matrix < 0).astype(int) + 1


def uniform_array(level, factors):
    """The smallest array here with `factors` columns, all at `level` levels."""
    if factors <= 1:
        return np.arange(1, level + 1).reshape(-1, 1)
    if field(level) is None:
        return None  # six levels: two orthogonal Latin squares of order 6 do not exist
    for power in (2, 3, 4):
        runs = level**power
        if runs > 36:
            break
        if level == 2 and runs == 16 and factors <= 11:
            return plackett_burman_12()[:, :factors]  # 12 runs beat 16
        if (runs - 1) // (level - 1) >= factors:
            return galois_array(level, power)[:, :factors]
    if level == 2 and factors <= 11:
        return plackett_burman_12()[:, :factors]
    return None


def orthogonal_array(levels):
    """An array with one column per factor; `levels` is each factor's number of levels.

    Factors are grouped by their number of levels, one array is built per group, and the
    groups are crossed: every run of one group's array meets every run of the next.
    """
    positions = {}
    for index, level in enumerate(levels):
        positions.setdefault(level, []).append(index)
    blocks = {}
    for level in sorted(positions):
        block = uniform_array(level, len(positions[level]))
        if block is None:
            raise ValueError(f"no orthogonal array for {len(positions[level])} factors "
                             f"at {level} levels")
        blocks[level] = block
    runs = int(np.prod([block.shape[0] for block in blocks.values()]))
    if runs > 36:
        raise ValueError(f"this combination of levels needs {runs} runs, more than 36")
    columns = [None] * len(levels)
    repeat, tile = runs, 1
    for level in sorted(positions):
        block = blocks[level]
        repeat //= block.shape[0]
        for offset, index in enumerate(positions[level]):
            columns[index] = np.tile(np.repeat(block[:, offset], repeat), tile)
        tile *= block.shape[0]
    return np.column_stack(columns)

## Libraries

In [ ]:
import itertools

import numpy as np
import pandas as pd
from orthogonal_array import orthogonal_array

pd.set_option("display.width", 120)

## The factors and their levels

In [ ]:
# One column per factor, its level names down the rows. A blank means that factor has
# fewer levels than the widest one.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Subject line", "Send time", "Image", "Button colour"]


def label(value):
    # A whole number read as 10.0 is written 10, as in the sheet.
    return str(int(value)) if isinstance(value, float) and value.is_integer() else str(value)


levels = {name: [label(v) for v in data[name] if not pd.isna(v) and str(v).strip()]
          for name in variables}
for name in variables:
    print(f"{name}: {len(levels[name])} levels -- {', '.join(levels[name])}")

## Orthogonal array (numbers)

In [ ]:
# Each row is one run of the experiment: which level of each factor to use. NumPy has
# no orthogonal-array function, so the construction is written out in
# orthogonal_array.py.
array = orthogonal_array([len(levels[name]) for name in variables])
runs = pd.RangeIndex(1, len(array) + 1, name="run")
numeric = pd.DataFrame(array, index=runs, columns=variables)
full = int(np.prod([len(levels[name]) for name in variables]))
print(f"\n{len(array)} runs instead of the {full} of every combination")
print(numeric)

## Check that the array is balanced

In [ ]:
# Every pair of factors should meet every pair of their levels equally often: that is
# what lets each main effect be estimated free of the others.
balanced = all(len(np.unique(pd.crosstab(numeric[a], numeric[b]).to_numpy())) == 1
               for a, b in itertools.combinations(variables, 2))
print(f"\nevery pair of factors is balanced: {balanced}")

## Orthogonal array (level names)

In [ ]:
named = pd.DataFrame({name: [levels[name][v - 1] for v in numeric[name]] for name in variables},
                     index=runs)
print()
print(named)

## What the result says

This method plans a test. It does not analyse results. An orthogonal array is a short list of versions in which the levels of each feature appear equally often and meet the levels of the other features in a balanced way.

The array has 12 runs, half of what every combination of the four features would need. Each feature's own effect can still be measured fairly.

There is a price. In the table of level names, a question with no image is always red, and a statement with no image is always blue. So the button colour is tangled up (confounded) with how subject line and image combine, and the two cannot be separated. Send the 12 versions if that risk is acceptable.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.